# Income classification: a full tabular pipeline

End-to-end binary classification on census-style demographic data, predicting
whether a person earns above a given income threshold.

The notebook runs the whole pipeline rather than jumping to a model: inspect
distributions, handle missing values and duplicates, encode nine categorical
columns by three different strategies depending on what each column *means*,
check for collinearity, then compare Logistic Regression against an SVM under
10-fold cross-validation and a grid search.

It closes with an unsupervised counterpoint, K-Means with `k=2` on the same
features and no access to the labels, to measure how much of the signal is
actually recoverable without supervision.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import mode
from sklearn.metrics.pairwise import pairwise_distances_argmin

# Path to the dataset. It is NOT committed to this repository.
# See ../data/README.md for the source and download instructions.
DATA_CSV = "../data/income.csv"


## Loading and first look

The dataset is not committed to this repository. See
[`../data/README.md`](../data/README.md) for the source and download steps.

In [ ]:
dataset = pd.read_csv(DATA_CSV)

#print out few head rows of the dataset
print(dataset.head())
print("\nDataset length:", len(dataset))
print("\nDataset initial shape:", dataset.shape)


In [ ]:
#check for distribution of numberical variables
print(dataset.describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].boxplot(dataset['age'])
axes[0].set_title('Age')
axes[0].set_ylabel('Years')

axes[1].boxplot(dataset['hours-per-week'])
axes[1].set_title('Hours per week')
axes[1].set_ylabel('Hours')

plt.tight_layout()
plt.show()

In [ ]:
print(dataset['income'].value_counts())

### Class balance

29.9% positive class. Not catastrophic, but skewed enough that accuracy alone
will be misleading: a model predicting "everyone is below the threshold" scores
70.1% without learning anything. That number becomes the baseline to beat, and it
comes back at the very end.

## Cleaning

Missing values sit in two columns only (`workclass`, `occupation`, ~1,400 rows
each) and overlap heavily. Dropping them costs 5% of the data; imputing a
categorical "unknown" would have been the alternative.

In [ ]:
#check for missing values
print(dataset.isna().sum(), "\n")

#Dealing with missing
dataset = dataset.dropna()
print(dataset.isna().sum(), "\n")
print("dataset length:", len(dataset))

In [ ]:
#Check for duplicated row
print(dataset.duplicated().any())
dataset = dataset.drop_duplicates()
print("dataset length:", len(dataset))

## Encoding categorical variables

Three different strategies, chosen per column by what the variable actually is:

- **`education` → ordinal integers.** The levels have a real order
  (Preschool < HS-grad < Bachelors < Doctorate), so one-hot encoding would throw
  that ordering away and spend 16 columns doing it.
- **`sex` → single binary column.** Two levels need one column, not two.
- **`workclass`, `marital-status`, `occupation`, `relationship`, `race` → one-hot.**
  No meaningful order, so any integer coding would invent one.

In [ ]:
# check value counts first
for col in ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex']:
    print(dataset[col].value_counts(), "\n")

In [ ]:
#Handle education variable
education_order = {
    'Preschool': 1, '1st-4th': 2, '5th-6th': 3, '7th-8th': 4, '9th': 5,
    '10th': 6, '11th': 7, '12th': 8, 'HS-grad': 9, 'Some-college': 10,
    'Assoc-voc': 11, 'Assoc-acdm': 12, 'Bachelors': 13, 'Masters': 14,
    'Prof-school': 15, 'Doctorate': 16
}
dataset['education'] = dataset['education'].replace(education_order)
dataset.head()

In [ ]:
#Binary sex variable
dataset['sex'] = dataset['sex'].replace({'Male': 0, 'Female': 1})
dataset.head()

In [ ]:
#Apply dummy code for the resrt (workclass, marital-status, occupation, relationship, race)
dataset = pd.get_dummies(dataset, columns=['workclass', 'marital-status', 'occupation', 'relationship', 'race'], dtype=int)
dataset.head()

## Correlation and collinearity

After encoding, the feature matrix is inspected for redundancy. One-hot encoding
a column whose categories overlap with another column's is the usual source of
trouble here.

In [ ]:
print("Shape (rows, columns):", dataset.shape)
dataset.dtypes
dataset.describe()

In [ ]:
corr = dataset.corr()
print(corr['income'].sort_values(ascending=False))

plt.figure(figsize=(12, 10))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation matrix (after pre-processing)")
plt.tight_layout()
plt.show()

`relationship_Husband` correlates 0.406 with the target but is almost a
restatement of `marital-status_Married` (0.459). The two encode nearly the same
fact about the same people. Dropping the weaker, redundant one removes the
collinearity without losing information.

In [ ]:
dataset = dataset.drop(columns=['relationship_Husband'])
corr = dataset.corr()
print(corr['income'].sort_values(ascending=False))

plt.figure(figsize=(12, 10))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation matrix (after pre-processing)")
plt.tight_layout()
plt.show()

In [ ]:
print("Shape (rows, columns):", dataset.shape)
print(dataset.head())

## Split and scale

Scaler is fitted on the **training split only** and then applied to the test
split. Fitting it on the full dataset first would leak test-set range information
into training.

In [ ]:
cols = [c for c in dataset.columns if c != 'income'] + ['income']
dataset = dataset[cols]

array = dataset.values
X = array[:, 0:-1]
y = array[:, -1].astype('int')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=123)

In [ ]:
# fit scaler on training data
norm = MinMaxScaler().fit(X_train)

# transform training data
X_train_norm = norm.transform(X_train)

# transform testing data
X_test_norm = norm.transform(X_test)

## Baseline models, default settings

In [ ]:
#Logistic Regression model
model = LogisticRegression(solver="liblinear")
model.fit(X_train_norm, y_train)
test_score = model.score(X_test_norm, y_test)
print("Testing Accuracy of LR:", test_score)

#SVM model
model = SVC()
model.fit(X_train_norm, y_train)
test_score = model.score(X_test_norm, y_test)
print("Testing Accuracy of SVC:", test_score)

## 10-fold cross-validation

A single train/test split gives one number with no sense of its variance.
Cross-validating on the training split gives a more honest estimate before any
tuning happens.

In [ ]:
#Define 10-fold-cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=2)

In [ ]:
# Evaluate the two models based on the average score
model = LogisticRegression(solver="liblinear")
results = cross_val_score(model, X_train_norm, y_train, cv=kfold)
print("Average Accuracy of LR:", results.mean())

model = SVC()
results = cross_val_score(model, X_train_norm, y_train, cv=kfold)
print("Average Accuracy of SVM:", results.mean())

## Hyperparameter search

In [ ]:
# Logistic Regression
from sklearn.model_selection import GridSearchCV

# fine tune parameters for lr model
grid_params_lr = {
    'penalty': ['l1', 'l2'],
    'C': [1, 10],
    'solver': ['saga', 'liblinear']
}

lr = LogisticRegression(max_iter=1000)
gs_lr_result = GridSearchCV(lr, grid_params_lr, cv=kfold).fit(X_train_norm, y_train)
print(gs_lr_result.best_score_)

In [ ]:
#SVM
grid_params_svc = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'C': [1, 10]
}

svc = SVC()
gs_svc_result = GridSearchCV(svc, grid_params_svc, cv=kfold).fit(X_train_norm, y_train)
print(gs_svc_result.best_score_)

## Test-set evaluation of the tuned models

In [ ]:
# Evaluate the trained Logistic Regression model using testing dataset
test_accuracy = gs_lr_result.best_estimator_.score(X_test_norm, y_test)
print("Accuracy in testing:", test_accuracy)

# check the parameter setting for the best selected model
gs_lr_result.best_params_

In [ ]:
# Evaluate the trained SVM model using testing dataset
test_accuracy = gs_svc_result.best_estimator_.score(X_test_norm, y_test)
print("Accuracy in testing:", test_accuracy)

# check the parameter setting for the best selected model
gs_svc_result.best_params_

### Supervised results

| Model | CV (10-fold) | Test accuracy | Best parameters |
|---|---|---|---|
| Logistic Regression, defaults | 0.8052 | 0.8101 | n/a |
| SVM (RBF), defaults | 0.7955 | 0.7976 | n/a |
| **Logistic Regression, tuned** | 0.8054 | **0.8110** | `C=1, penalty=l1, solver=saga` |
| SVM, tuned | 0.8061 | 0.8045 | `C=10, kernel=rbf` |

Two things worth reading carefully.

**Tuning bought almost nothing.** Logistic Regression went from 0.8101 to 0.8110
on test, one tenth of a percentage point, across a grid of 8 configurations.
The SVM's cross-validation score improved (0.7955 to 0.8061) yet its *test* score
moved barely at all. When a grid search produces a flat landscape like this, the
model is not what is limiting performance; the features are.

**The simpler model won.** An L1-penalised linear model beat an RBF-kernel SVM on
the test set, while training in a fraction of the time. The L1 penalty is also
doing feature selection for free. On 40 mostly-sparse one-hot columns that is
exactly the right inductive bias, and it suggests the decision boundary here is
close to linear.

## Unsupervised counterpoint: K-Means

The same scaled features, `k=2`, no labels. The question is how much of the
income split is visible in the geometry of the data alone.

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=0).fit(X_train_norm)

In [ ]:
kmeans_labels = kmeans.labels_
unique_labels, unique_counts = np.unique(kmeans_labels, return_counts=True)
dict(zip(unique_labels, unique_counts))

### Cluster prototypes

Rather than reading cluster centroids (which are averages of one-hot columns and
so correspond to no real person), this takes the *nearest actual record* to each
centroid, giving an interpretable representative of each cluster.

In [ ]:
closest = pairwise_distances_argmin(kmeans.cluster_centers_, X_train_norm)

feature_names = [c for c in dataset.columns if c != 'income']
train_df = pd.DataFrame(X_train, columns=feature_names)

prototypes = train_df.iloc[closest]
prototypes

In [ ]:
train_income = y_train[closest]
for i, row_idx in enumerate(closest):
    row = train_df.iloc[row_idx]
    active = [c for c in feature_names if c not in ['age', 'education', 'sex', 'hours-per-week'] and row[c] == 1]
    print(f"Prototype {i}: age={row['age']}, education={row['education']}, sex={row['sex']}, "
          f"hours={row['hours-per-week']}, true income={train_income[i]}")
    print("  active categories:", active)

The two prototypes separate almost exactly as you would guess: a
36-year-old in clerical work, not married, below the threshold; and a 43-year-old
in an executive/managerial role, married, working 45 hours, above it. K-Means
found the married/managerial axis unsupervised.

## Does the clustering actually predict income?

In [ ]:
kmeans_test_labels = kmeans.predict(X_test_norm)

acc_raw = accuracy_score(y_test, kmeans_test_labels)
acc_flipped = accuracy_score(y_test, 1 - kmeans_test_labels)

print("k means prediction accuracy (raw):", acc_raw)
print("k means prediction accuracy (flipped):", acc_flipped)
print("k means prediction accuracy (best):", max(acc_raw, acc_flipped))

In [ ]:
majority_baseline = max(np.mean(y_test == 0), np.mean(y_test == 1))
print("Majority-class baseline accuracy:", majority_baseline)

### Clustering results

| | Accuracy |
|---|---|
| K-Means, cluster to label as-is | **0.7205** |
| K-Means, labels flipped | 0.2795 |
| Majority-class baseline | 0.6964 |
| Tuned Logistic Regression | 0.8110 |

The honest reading: 0.7205 looks respectable until you put the majority-class
baseline next to it. K-Means beats "always predict the majority class" by **2.4
percentage points**, which is real but small. Supervision is worth a further 9
points on top.

The flipped figure (0.2795 = 1 - 0.7205) is included deliberately: cluster labels
from K-Means are arbitrary identifiers, so reporting accuracy without checking
both cluster-to-label mappings is a way to accidentally publish a number that is
exactly backwards.

So the income boundary is *partly* a natural cluster boundary in this feature
space, but mostly it is not, and that gap is what the supervised models are
being paid for.